In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.agents import create_agent
from langchain.agents import AgentState
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.tools import tool
from langchain.tools import ToolRuntime
from langchain.messages import HumanMessage
from langchain.messages import ToolMessage 
from langgraph.types import Command
from langchain_community.utilities import SQLDatabase

from typing import Dict, Any
from tavily import TavilyClient

In [2]:
# MCP

client = MultiServerMCPClient(
    {
        "travel_server": {
                "transport": "streamable_http",
                "url": "https://mcp.kiwi.com"
            }
    }
)

tools = await client.get_tools()

In [3]:
tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

In [14]:
import kagglehub
from pathlib import Path


target_dir = Path("../assets/chinook") 
target_dir.mkdir(parents=True, exist_ok=True)

path = kagglehub.dataset_download(
    "ranasabrii/chinook",
    output_dir=str(target_dir)
)

print("Path to dataset files:", path)

100%|██████████| 438k/438k [00:00<00:00, 1.42MB/s]

Extracting files...
Path to dataset files: ../assets/chinook


In [ ]:

db = SQLDatabase.from_uri("sqlite:///../assets/chinook/Chinook.db")

@tool
def query_playlist_db(query: str) -> str:

    """Query the database for playlist information"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error querying database: {e}"

In [16]:
# create state

class WeddingState(AgentState):
    origin: str
    destination: str
    guest_count: str
    genre: str

In [17]:
# prompts

SYSTEM_PROMPT_TRAVEL = """
You are a travel agent. Search for flights to the desired destination wedding location.
    You are not allowed to ask any more follow up questions, you must find the best flight options based on the following criteria:
    - Price (lowest, economy class)
    - Duration (shortest)
    - Date (time of year which you believe is best for a wedding at this location)
    To make things easy, only look for one ticket, one way.
    You may need to make multiple searches to iteratively find the best options.
    You will be given no extra information, only the origin and destination. It is your job to think critically about the best options.
    Once you have found the best options, let the user know your shortlist of options.
"""

SYSTEM_PROMPT_VENUE = """
    You are a venue specialist. Search for venues in the desired location, and with the desired capacity.
    You are not allowed to ask any more follow up questions, you must find the best venue options based on the following criteria:
    - Price (lowest)
    - Capacity (exact match)
    - Reviews (highest)
    You may need to make multiple searches to iteratively find the best options.
"""

SYSTEM_PROMPT_PLAYLIST = """
    You are a playlist specialist. Query the sql database and curate the perfect playlist for a wedding given a genre.
    Once you have your playlist, calculate the total duration and cost of the playlist, each song has an associated price.
    If you run into errors when querying the database, try to fix them by making changes to the query.
    Do not come back empty handed, keep trying to query the db until you find a list of songs.
    You may need to make multiple queries to iteratively find the best options.
"""



In [19]:
# create subagents

MODEL_NAME = "openai:gpt-5-nano"

travel_agent = create_agent(
    model=MODEL_NAME,
    tools=tools,
    system_prompt=SYSTEM_PROMPT_TRAVEL,
    
)

venue_agent = create_agent(
    model=MODEL_NAME,
    tools=[web_search],
    system_prompt=SYSTEM_PROMPT_VENUE,
    
)

playlist_agent = create_agent(
    model=MODEL_NAME,
    tools=[query_playlist_db],
    system_prompt=SYSTEM_PROMPT_PLAYLIST,
    
)

In [20]:
# wrappers 

@tool
async def search_flights(runtime: ToolRuntime) -> str:
    """Travel agent searches for flights to the desired destination wedding location."""
    origin = runtime.state["origin"]
    destination = runtime.state["destination"]
    response = await travel_agent.ainvoke({"messages": [HumanMessage(content=f"Find flights from {origin} to {destination}")]})
    return response['messages'][-1].content

@tool
def search_venues(runtime: ToolRuntime) -> str:
    """Venue agent chooses the best venue for the given location and capacity."""
    destination = runtime.state["destination"]
    capacity = runtime.state["guest_count"]
    query = f"Find wedding venues in {destination} for {capacity} guests"
    response = venue_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def suggest_playlist(runtime: ToolRuntime) -> str:
    """Playlist agent curates the perfect playlist for the given genre."""
    genre = runtime.state["genre"]
    query = f"Find {genre} tracks for wedding playlist"
    response = playlist_agent.invoke({"messages": [HumanMessage(content=query)]})
    return response['messages'][-1].content

@tool
def update_state(origin: str, destination: str, guest_count: str, genre: str, runtime: ToolRuntime) -> str:
    """Update the state when you know all of the values: origin, destination, guest_count, genre"""
    return Command(update={
        "origin": origin, 
        "destination": destination, 
        "guest_count": guest_count, 
        "genre": genre, 
        "messages": [ToolMessage("Successfully updated state", tool_call_id=runtime.tool_call_id)]}
        )


In [21]:
coordinator = create_agent(
    model="gpt-5-nano",
    tools=[search_flights, search_venues, suggest_playlist, update_state],
    state_schema=WeddingState,
    system_prompt="""
    You are a wedding coordinator. Delegate tasks to your specialists for flights, venues and playlists.
    First find all the information you need to update the state. Once that is done you can delegate the tasks.
    Once you have received their answers, coordinate the perfect wedding for me.
    """
)


In [ ]:
# Test

response = await coordinator.ainvoke(
    {
        "messages": [HumanMessage(content="I'm from London and I'd like a wedding in Paris for 100 guests, jazz-genre")],
    }
)

In [ ]:
print(response["messages"][-1].content)

Fantastic — I’ve pulled together the core pieces for a 100-guest jazz wedding in Paris from London. Here’s a clear, actionable plan based on what you shared.

1) Flights (for guests from London to Paris)
Top easy options to consider (mid-June date, economy)
- Cheapest / short duration: STN (Stansted) to CDG (Paris)
  - 15/06/2026, 15:35 – 17:45, 1h10m
  - Price: 53 EUR
  - Booking: https://on.kiwi.com/IbOYy0
- Backup affordable: LTN (Luton) to CDG
  - 15/06/2026, 19:40 – 22:00, 1h20m
  - Price: 53 EUR
  - Booking: https://on.kiwi.com/fXCZ7B
- Shortest alternative: SEN (Southend) to CDG
  - 15/06/2026, 11:10 – 13:20, 1h10m
  - Price: 89 EUR
  - Booking: https://on.kiwi.com/1SpFP0

Notes:
- STN-CDG at 53 EUR is the standout for price + travel time.
- For central London access, LHR→ORY or LGW→CDG options exist (slightly higher cost but easier travel from central areas).

Next steps I can take for you:
- Create a group-flight block with preferred London airports and exact travel date windo